# YOLO для изображений: блочный и переносимый шаблон

Этот ноутбук сделан как набор простых блоков, которые можно переносить в другой проект почти без изменений.

Идея:
- в начале задаём пути и параметры;
- дальше идут небольшие функции с понятными задачами;
- затем отдельно показываем инференс на одном изображении и на папке с изображениями;
- в конце есть блок с краткой адаптацией под свои данные.

Все примеры используют лёгкую модель YOLO.

## Структура ноутбука

1. Импорт библиотек  
2. Настройки путей и параметров  
3. Универсальные функции  
4. Загрузка модели  
5. Инференс на одном изображении  
6. Инференс на папке изображений  
7. Визуализация и сохранение результатов  
8. Что менять при переносе в другой проект

In [ ]:
# =========================
# БЛОК 1. Импорты
# Этот блок можно переносить как есть.
# =========================

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from ultralytics import YOLO

In [ ]:
# =========================
# БЛОК 2. Пути и основные параметры
# Менять обычно нужно именно этот блок.
# =========================

# Лёгкая модель для быстрого старта.
MODEL_WEIGHTS = "yolov8n.pt"

# Пути к данным.
# Замените на свои реальные пути.
IMAGE_PATH = Path("data/images/example.jpg")
IMAGE_DIR = Path("data/images")
OUTPUT_DIR = Path("outputs/image_inference")

# Параметры инференса.
IMG_SIZE = 640
CONF_THRESHOLD = 0.25
DEVICE = "cpu"   # если есть GPU и всё настроено, можно указать '0'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_WEIGHTS =", MODEL_WEIGHTS)
print("IMAGE_PATH    =", IMAGE_PATH)
print("IMAGE_DIR     =", IMAGE_DIR)
print("OUTPUT_DIR    =", OUTPUT_DIR)

In [ ]:
# =========================
# БЛОК 3. Вспомогательные функции
# Они небольшие и легко вытаскиваются в любой другой notebook или .py файл.
# =========================

def show_image(image, title="Image", figsize=(10, 6)):
    """Показ изображения через matplotlib.

    Поддерживает:
    - путь к файлу,
    - PIL.Image,
    - numpy array в формате OpenCV (BGR) или RGB.
    """
    plt.figure(figsize=figsize)

    if isinstance(image, (str, Path)):
        image = Image.open(image)
        image = np.array(image)

    if isinstance(image, Image.Image):
        image = np.array(image)

    if isinstance(image, np.ndarray):
        # Если массив выглядит как цветное изображение OpenCV, переводим BGR -> RGB.
        if image.ndim == 3 and image.shape[2] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()


def ensure_file_exists(path):
    """Простая проверка существования файла."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")
    return path


def ensure_dir_exists(path):
    """Простая проверка существования папки."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Папка не найдена: {path}")
    return path


def load_yolo_model(weights_path):
    """Загрузка модели YOLO из готовых весов."""
    model = YOLO(weights_path)
    return model


def predict_single_image(model, image_path, conf=0.25, imgsz=640, device="cpu"):
    """Инференс на одном изображении.

    Возвращает объект результата YOLO.
    """
    image_path = ensure_file_exists(image_path)
    results = model.predict(
        source=str(image_path),
        conf=conf,
        imgsz=imgsz,
        device=device,
        verbose=False
    )
    return results[0]


def predict_image_folder(model, image_dir, conf=0.25, imgsz=640, device="cpu", save=True, project="outputs", name="batch_inference"):
    """Инференс на папке с изображениями.

    YOLO сама умеет:
    - пройтись по всем изображениям,
    - сохранить отрисованные результаты.
    """
    image_dir = ensure_dir_exists(image_dir)
    results = model.predict(
        source=str(image_dir),
        conf=conf,
        imgsz=imgsz,
        device=device,
        save=save,
        project=project,
        name=name,
        exist_ok=True,
        verbose=False
    )
    return results


def extract_detections_to_table(result):
    """Преобразование детекций в простой список словарей.

    Это удобно, если дальше нужно:
    - фильтровать объекты,
    - считать статистику,
    - перекладывать данные в DataFrame.
    """
    detections = []

    if result.boxes is None:
        return detections

    boxes = result.boxes
    class_ids = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else []
    confidences = boxes.conf.cpu().numpy() if boxes.conf is not None else []
    xyxy = boxes.xyxy.cpu().numpy() if boxes.xyxy is not None else []

    for cls_id, conf, coords in zip(class_ids, confidences, xyxy):
        detections.append(
            {
                "class_id": int(cls_id),
                "class_name": result.names[int(cls_id)],
                "confidence": float(conf),
                "x1": float(coords[0]),
                "y1": float(coords[1]),
                "x2": float(coords[2]),
                "y2": float(coords[3]),
            }
        )

    return detections

In [ ]:
# =========================
# БЛОК 4. Загрузка модели
# Обычно этот блок переносится без изменений.
# =========================

model = load_yolo_model(MODEL_WEIGHTS)
model

## Быстрый пример на одном изображении

Ниже самый частый сценарий:
1. указать путь к картинке;
2. запустить инференс;
3. показать итоговую картинку с боксами;
4. при необходимости разобрать детекции в таблицу.

In [ ]:
# =========================
# БЛОК 5. Инференс на одном изображении
# =========================

# Раскомментируйте, когда будет реальное изображение:
# result = predict_single_image(
#     model=model,
#     image_path=IMAGE_PATH,
#     conf=CONF_THRESHOLD,
#     imgsz=IMG_SIZE,
#     device=DEVICE
# )

# Если результат получен, можно вывести картинку:
# plotted = result.plot()
# show_image(plotted, title="Predicted image")

print("Готово. Подставьте реальный IMAGE_PATH и снимите комментарии в этом блоке.")

In [ ]:
# =========================
# БЛОК 6. Разбор детекций в удобный список
# Этот блок полезен, если дальше будет логика по объектам.
# =========================

# detections = extract_detections_to_table(result)
# detections[:5]

print("После запуска инференса здесь можно получить список объектов с координатами и confidence.")

## Пакетная обработка папки с изображениями

Этот режим удобен, если нужно быстро прогнать много файлов и сохранить результаты с отрисованными рамками.

In [ ]:
# =========================
# БЛОК 7. Инференс на папке изображений
# =========================

# results = predict_image_folder(
#     model=model,
#     image_dir=IMAGE_DIR,
#     conf=CONF_THRESHOLD,
#     imgsz=IMG_SIZE,
#     device=DEVICE,
#     save=True,
#     project=str(OUTPUT_DIR),
#     name="folder_run"
# )

print("Подставьте реальную папку IMAGE_DIR и снимите комментарии для пакетной обработки.")

In [ ]:
# =========================
# БЛОК 8. Просмотр сохранённых результатов
# Этот блок удобно использовать отдельно.
# =========================

saved_dir = OUTPUT_DIR / "folder_run"
print("Ожидаемая папка с сохранёнными изображениями:", saved_dir)

# Пример просмотра первых файлов:
# saved_images = sorted(saved_dir.glob("*"))
# print("Найдено файлов:", len(saved_images))
# if saved_images:
#     show_image(saved_images[0], title=f"Saved prediction: {saved_images[0].name}")

## Что обычно меняют при переносе в другой проект

Чаще всего надо поменять только:
- `MODEL_WEIGHTS`
- `IMAGE_PATH`
- `IMAGE_DIR`
- `OUTPUT_DIR`
- `CONF_THRESHOLD`
- `IMG_SIZE`

Остальные блоки можно оставлять почти без изменений.

## Идеи для расширения

Этот шаблон легко расширить:
- фильтрация только нужных классов;
- подсчёт количества объектов;
- экспорт результатов в CSV;
- объединение с трекингом;
- запуск по видео;
- дообучение на своём датасете.

Для видео и дообучения будет удобнее вынести отдельный ноутбук.